# LLM Chain 

In [4]:
from dotenv import load_dotenv 
import os 

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser


template1 = "explain {question} in simple terms."
template2 = "summarize the following review and extract the sentiment: {text} return the result in JSON format with keys 'summary' and 'sentiment'"

prompt1 = ChatPromptTemplate.from_template(template1)
prompt2 = ChatPromptTemplate.from_template(template2)
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.1)

parser_text = StrOutputParser()
parser_json = JsonOutputParser()

chain1 = prompt1 | llm | parser_text 

result1 = chain1.invoke({
    "question": "What is LangChain?"
})

chain2 = prompt2 | llm | parser_json

result2 = chain2.invoke({
    "text" : "I recently bought this product and it was amazing! The quality is top-notch and it exceeded my expectations. I would highly recommend it to anyone looking for a great purchase."
})

In [6]:
print(result1)
print(result2)

LangChain is a software library that helps you build apps that use language models (like GPT-4) by connecting them to data, tools, and your own logic. In simple terms, it gives you building blocks to create smarter AI helpers without writing a lot of boilerplate.

Key ideas:
- Chains: step-by-step processes where the output of one prompt becomes the input for the next (like a little workflow).
- Agents: components that can decide what to do next and call external tools or APIs (e.g., search, databases, calculators) to complete a task.
- Tools/Plugins: connections to external systems (APIs, databases, file systems) that the model can use.
- Memory: keeps track of prior context or conversation history to make interactions smoother.
- Prompts and parsing: templates and ways to structure and extract data from model responses.

What it’s for:
- Building chatbots and virtual assistants
- Answering questions by querying data sources
- Automating tasks that require both reasoning and real-worl

In [10]:
type(result2)
result2.get("sentiment")

'very positive'

# Sequential chain 

```
topic
 ↓
outline prompt
 ↓
LLM
 ↓
outline
 ↓
article prompt
 ↓
LLM
 ↓
article
```

In [ ]:
# step 1 
outline_template = "Create an outline for an article about {topic} in 5 bullet points and 100 words."
outline_prompt = ChatPromptTemplate.from_template(outline_template)

# step2 
article_template = "Write an article based on the following outline: {outline} with 500 words."
article_prompt = ChatPromptTemplate.from_template(article_template)

# chains 

outline_chain = outline_prompt | llm | parser_text
article_chain = article_prompt | llm | parser_text

# sequential chain 
# 
chain =  outline_chain | (lambda outline : {'outline' : outline}) | article_chain

In [14]:
result = chain.invoke({
    'topic' : "The benefits of using AI in business"
})
print(result)

Automation is reshaping how organizations operate, turning repetitive tasks into well-orchestrated processes and freeing human workers to focus on strategic activities, creative problem solving, and high‑value customer interactions. Robotic process automation, workflow orchestration, and AI-powered assistants handle routine data gathering, invoice processing, scheduling, and status updates, while people apply judgment, empathy, and leadership where it counts. The outcome is faster throughput, fewer human errors, and a more engaging work environment where employees can contribute at a higher level and customers experience quicker, more reliable service.

Data‑driven decision making is the backbone of modern strategy. By synthesizing vast datasets from across operations, markets, and customer touchpoints, organizations achieve greater accuracy in forecasting and risk assessment than traditional methods allow. AI, machine learning, and advanced analytics reveal hidden patterns, correlatio

# router chain

In [25]:
# chain 1 

math_prompt = ChatPromptTemplate.from_template(
    "You are an expert in math. Solve : {question}. \n Return the answer only."
)

math_chain = math_prompt | llm | parser_text

# chain 2 

history_prompt = ChatPromptTemplate.from_template(
     "You are a historian. Answer: {question} Return the answer with 50 words."
)
history_chain = history_prompt | llm | parser_text

In [26]:
math_chain

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an expert in math. Solve : {question}. \n Return the answer only.'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000251E0CDAFD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000251E11A9E90>, root_client

In [24]:
router_prompt = ChatPromptTemplate.from_template(
"""
You are a router.

Decide which chain should answer the question.

Return ONLY one word:
- "math"
- "history"

Question: {question}
"""
)

router_chain = router_prompt | llm | parser_text

In [ ]:
router_with_input = {
    "route": router_chain,
    "question": lambda x: x["question"]
} # converting into a RunnableParallel object

In [30]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "math" in x["route"], math_chain),
    (lambda x : 'history' in x["route"], history_chain),
    history_chain  # default
)

final_chain = router_with_input | branch

In [34]:
res = final_chain.invoke({
    'question' : "Who was the first president of the United States?"})

'George Washington served as the first president of the United States, guiding the new republic from 1789 to 1797. A transformative figure in American history, he championed republican virtue, helped establish constitutional norms, and shaped the executive office. His leadership stabilized national unity after independence and laid foundational practices everywhere.'

# explicit interpretation

In [ ]:
from langchain_core.runnables import RunnableLambda , RunnableParallel

router_with_input = RunnableParallel(
    route=router_chain,
    question=RunnableLambda(lambda x: x["question"])
)

In [40]:
router_with_input.invoke(  {
    'question' : '2 + 2 = ?'
})

{'route': 'math', 'question': '2 + 2 = ?'}

In [41]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda output_router: "math" in output_router["route"], math_chain),
    (lambda output_router: 'history' in output_router["route"], history_chain),
    history_chain  # default
)

final_chain = router_with_input | branch

In [42]:
final_chain.invoke(
    {
        'question' : '2 + 2 = ?'
    }
)

'4'

LangChain sẽ cố ép object thành Runnable.
Đó là lý do:

```
prompt | llm | parser
dict | branch
lambda | chain
```

đều chạy được.